<a href="https://colab.research.google.com/github/ehsanre1376/YouTube-DownLoader-To-Colab/blob/main/telBot1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 1. Install Dependencies
# ============================================================
!pip install yt-dlp python-telegram-bot --quiet

# ============================================================
# 2. Import Libraries
# ============================================================
import logging
import os
import shutil
import uuid
import asyncio
import re # For basic URL check
from telegram import Update, InputFile
from telegram.ext import Application, CommandHandler, MessageHandler, filters, ContextTypes
from telegram.constants import ParseMode, ChatAction # Import ChatAction
from telegram.error import TelegramError
from yt_dlp import YoutubeDL
from yt_dlp.utils import DownloadError

# ============================================================
# 3. Configuration
# ============================================================
TELEGRAM_BOT_TOKEN = "7668113099:AAHwpP6FHOlifpSqhFZTj03Zc6r0k2HrR8o"  # Your Bot Token
DOWNLOAD_BASE_DIR = "/content/downloads"  # Base directory for downloads in Colab
MAX_TELEGRAM_FILE_SIZE_MB = 1900 # Telegram Bot API limit is 2000MB, stay slightly below
MAX_TELEGRAM_FILE_SIZE_BYTES = MAX_TELEGRAM_FILE_SIZE_MB * 1024 * 1024

# ============================================================
# 4. Logging Setup
# ============================================================
logging.basicConfig(
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s", level=logging.INFO
)
# Set higher logging level for httpx and telegram to avoid excessive noise
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("telegram").setLevel(logging.WARNING) # Added for telegram library
logging.getLogger("telegram.ext").setLevel(logging.INFO) # Keep ext logging for bot actions
logger = logging.getLogger(__name__)

# ============================================================
# 5. Helper Functions
# ============================================================

async def cleanup(download_dir: str, zip_path: str = None):
    """Removes the download directory and the zip file."""
    logger.info(f"Cleaning up {download_dir} and {zip_path}")
    try:
        if download_dir and os.path.exists(download_dir) and os.path.isdir(download_dir):
            shutil.rmtree(download_dir)
            logger.info(f"Removed directory: {download_dir}")
    except OSError as e:
        logger.error(f"Error removing directory {download_dir}: {e}")
    except Exception as e:
         logger.error(f"Unexpected error removing directory {download_dir}: {e}")


    try:
        if zip_path and os.path.exists(zip_path) and os.path.isfile(zip_path):
            os.remove(zip_path)
            logger.info(f"Removed zip file: {zip_path}")
    except OSError as e:
        logger.error(f"Error removing zip file {zip_path}: {e}")
    except Exception as e:
        logger.error(f"Unexpected error removing zip file {zip_path}: {e}")


def is_youtube_url(url: str) -> bool:
    """Basic check if the string looks like a YouTube URL."""
    # This is a simple check, yt-dlp will do the definitive validation
    if not isinstance(url, str):
        return False
    return bool(re.match(r'(https?://)?(www\.)?(youtube\.com|youtu\.be)/', url))

# ============================================================
# 6. Core Download and Upload Logic
# ============================================================

async def download_and_send(update: Update, context: ContextTypes.DEFAULT_TYPE, url: str):
    """Downloads video(s), sends them, zips, sends zip, and cleans up."""
    if not update or not update.effective_chat or not update.message:
        logger.error("Update object is missing necessary attributes.")
        return

    chat_id = update.effective_chat.id
    message_id = update.message.message_id
    processing_message = None # Initialize

    try:
        # Send initial status message
        processing_message = await context.bot.send_message(chat_id=chat_id, text="🔄 Processing your request...", reply_to_message_id=message_id)
    except TelegramError as e:
        logger.error(f"Failed to send initial processing message to chat {chat_id}: {e}")
        # Cannot proceed without informing the user
        return
    except Exception as e:
         logger.error(f"Unexpected error sending initial message to chat {chat_id}: {e}", exc_info=True)
         return # Stop if we can't even send the first message

    # Create a unique directory for this download request
    request_id = str(uuid.uuid4())
    download_dir = os.path.join(DOWNLOAD_BASE_DIR, request_id)
    zip_path = None # Initialize zip_path

    try:
        os.makedirs(download_dir, exist_ok=True)
        logger.info(f"Created download directory: {download_dir}")

        await context.bot.edit_message_text(chat_id=chat_id, message_id=processing_message.message_id, text="📥 Starting download...")

        # Configure yt-dlp options
        ydl_opts = {
            'format': 'bestvideo[ext=mp4][height<=1080]+bestaudio[ext=m4a]/best[ext=mp4][height<=1080]/best[ext=mp4]/best',
            'outtmpl': os.path.join(download_dir, '%(title)s [%(id)s].%(ext)s'),
            'writesubtitles': True,
            'subtitleslangs': ['en', 'fa'], # English and Persian
            'subtitlesformat': 'srt',
            'writedescription': False,
            'writeinfojson': False,
            'writeannotations': False,
            'noplaylist': False, # Ensure playlists are downloaded
            'ignoreerrors': True, # Skip unavailable videos in playlists
            'quiet': True, # Suppress console output from yt-dlp itself
            'noprogress': True, # Don't show progress bars in console
            'postprocessors': [{ # Ensure final output is mp4 if merging occurred
                'key': 'FFmpegVideoConvertor',
                'preferedformat': 'mp4',
            }],
            'concurrent_fragment_downloads': 5, # Speed up fragment downloads
            'retries': 10, # Retry downloads on network issues
            'socket_timeout': 30, # Timeout for network operations
             'verbose': False, # Set to True for detailed yt-dlp debugging
            # Optional: Add progress hook later if needed for detailed Telegram updates
            # 'progress_hooks': [lambda d: progress_hook(d, context, chat_id, processing_message.message_id)],
        }

        # Execute download using asyncio compatible run_in_executor
        loop = asyncio.get_running_loop()
        try:
            logger.info(f"Starting yt-dlp download for {url} in executor")
            # Run the blocking yt-dlp download in a separate thread
            await loop.run_in_executor(None, lambda: YoutubeDL(ydl_opts).download([url]))
            logger.info(f"Download finished for URL: {url}")
        except DownloadError as e:
            # Specific yt-dlp errors (network, extraction, etc.)
            error_message = str(e).split('\n')[-1] # Try to get a concise error
            logger.error(f"yt-dlp download error for {url}: {error_message}")
            await context.bot.edit_message_text(
                chat_id=chat_id,
                message_id=processing_message.message_id,
                text=f"❌ Failed to download. Error: {error_message[:100]}{'...' if len(error_message)>100 else ''}" # Keep it short
            )
            # Cleanup happens in finally block
            return
        except Exception as e:
             # Catch other potential exceptions during download execution
            logger.error(f"Unexpected error during download execution for {url}: {e}", exc_info=True)
            await context.bot.edit_message_text(chat_id=chat_id, message_id=processing_message.message_id, text=f"❌ An unexpected error occurred during download.")
            # Cleanup happens in finally block
            return

        # Check if anything was downloaded
        try:
            downloaded_files = os.listdir(download_dir)
        except FileNotFoundError:
             logger.warning(f"Download directory {download_dir} seems to be missing after download attempt.")
             downloaded_files = []
        except Exception as e:
            logger.error(f"Error listing files in {download_dir}: {e}")
            downloaded_files = []

        if not downloaded_files:
            logger.warning(f"No files found in {download_dir} after download attempt for {url}. It might be unavailable or skipped.")
            await context.bot.edit_message_text(chat_id=chat_id, message_id=processing_message.message_id, text="⚠️ Download finished, but no files were obtained. The video/playlist might be unavailable, private, or empty.")
             # Cleanup happens in finally block
            return

        await context.bot.edit_message_text(chat_id=chat_id, message_id=processing_message.message_id, text="⬆️ Preparing to upload files...")

        # --- Send Individual MP4 Files ---
        video_files_sent = 0
        files_to_zip = []
        for filename in sorted(downloaded_files):
            file_path = os.path.join(download_dir, filename)
            if not os.path.isfile(file_path): # Skip directories if any created unexpectedly
                 logger.warning(f"Skipping non-file item in download dir: {filename}")
                 continue

            files_to_zip.append(filename) # Add all files (videos, srt) to be zipped

            if filename.lower().endswith(".mp4"):
                try:
                    file_size = os.path.getsize(file_path)
                except OSError as e:
                    logger.error(f"Could not get size of {file_path}: {e}. Skipping.")
                    continue

                if file_size > MAX_TELEGRAM_FILE_SIZE_BYTES:
                    logger.warning(f"Skipping direct upload of {filename} (size {file_size / (1024*1024):.2f} MB) - exceeds limit.")
                    await context.bot.send_message(chat_id=chat_id, text=f"⚠️ Video '{filename}' is too large ({file_size / (1024*1024):.2f} MB) for direct upload. It will be in the ZIP.", reply_to_message_id=message_id)
                    continue # Skip sending this large file individually

                try:
                    await context.bot.edit_message_text(chat_id=chat_id, message_id=processing_message.message_id, text=f"⬆️ Uploading video: {filename}...")
                    logger.info(f"Attempting to send video: {filename}")
                    await context.bot.send_chat_action(chat_id=chat_id, action=ChatAction.UPLOAD_VIDEO)
                    with open(file_path, 'rb') as video_file:
                        # Use increased timeouts specifically for file uploads
                        await context.bot.send_video(
                            chat_id=chat_id, video=video_file, caption=filename,
                            read_timeout=300, write_timeout=300, connect_timeout=60, pool_timeout=300
                        )
                    logger.info(f"Successfully sent video: {filename}")
                    video_files_sent += 1
                    await asyncio.sleep(2) # Increased delay between uploads
                except TelegramError as e:
                    logger.error(f"Failed to send video {filename}: {e}")
                    if "Too Large" in str(e):
                         await context.bot.send_message(chat_id=chat_id, text=f"❌ Video '{filename}' failed to upload (too large). It will be in the ZIP file.", reply_to_message_id=message_id)
                    else:
                         await context.bot.send_message(chat_id=chat_id, text=f"❌ Failed to send video '{filename}'. Error: {e}. It will still be in the ZIP file.", reply_to_message_id=message_id)
                except Exception as e:
                     logger.error(f"Unexpected error sending video {filename}: {e}", exc_info=True)
                     await context.bot.send_message(chat_id=chat_id, text=f"❌ Unexpected error sending video '{filename}'. It will still be in the ZIP file.", reply_to_message_id=message_id)

        if video_files_sent > 0:
             await context.bot.send_message(chat_id=chat_id, text=f"✅ Sent {video_files_sent} video file(s).", reply_to_message_id=message_id)
        else:
             await context.bot.send_message(chat_id=chat_id, text="ℹ️ No video files were sent individually (none downloaded/eligible or all failed/too large). Check the ZIP.", reply_to_message_id=message_id)


        # --- Create and Send ZIP Archive ---
        if not files_to_zip:
            logger.warning(f"No files were marked for zipping in {download_dir}.")
            await context.bot.edit_message_text(chat_id=chat_id, message_id=processing_message.message_id, text="ℹ️ No files found to create a ZIP archive.")
            # Cleanup happens in finally block
            return

        await context.bot.edit_message_text(chat_id=chat_id, message_id=processing_message.message_id, text="⚙️ Creating ZIP archive...")

        zip_filename_base = f"youtube_download_{request_id}"
        zip_path_base = os.path.join(DOWNLOAD_BASE_DIR, zip_filename_base) # Path without .zip initially

        try:
            logger.info(f"Creating zip archive: {zip_path_base}.zip from {download_dir}")
            # Run the blocking zip operation in an executor thread
            loop = asyncio.get_running_loop()
            zip_path = await loop.run_in_executor(None, lambda: shutil.make_archive(zip_path_base, 'zip', download_dir))
            logger.info(f"ZIP archive created: {zip_path}")
        except Exception as e:
            logger.error(f"Failed to create ZIP archive: {e}", exc_info=True)
            await context.bot.edit_message_text(chat_id=chat_id, message_id=processing_message.message_id, text=f"❌ Failed to create ZIP file. Error: {e}")
            # Cleanup happens in finally block
            return

        # Check zip size *before* attempting upload
        try:
            zip_size = os.path.getsize(zip_path)
        except OSError as e:
             logger.error(f"Could not get size of zip file {zip_path}: {e}")
             await context.bot.edit_message_text(chat_id=chat_id, message_id=processing_message.message_id, text=f"❌ Error checking ZIP file size.")
             # Cleanup happens in finally block
             return

        await context.bot.edit_message_text(chat_id=chat_id, message_id=processing_message.message_id, text=f"⬆️ Uploading ZIP archive ({zip_size / (1024*1024):.2f} MB)...")

        if zip_size > MAX_TELEGRAM_FILE_SIZE_BYTES:
             logger.warning(f"ZIP file {zip_path} (size {zip_size / (1024*1024):.2f} MB) is too large for Telegram.")
             await context.bot.send_message(chat_id=chat_id, text=f"⚠️ The final ZIP archive is too large ({zip_size / (1024*1024):.2f} MB) to upload via Telegram.", reply_to_message_id=message_id)
             await context.bot.edit_message_text(chat_id=chat_id, message_id=processing_message.message_id, text="✅ Done (individual files sent, ZIP too large).")
        else:
            try:
                await context.bot.send_chat_action(chat_id=chat_id, action=ChatAction.UPLOAD_DOCUMENT)
                with open(zip_path, 'rb') as zip_file:
                    await context.bot.send_document(
                        chat_id=chat_id, document=zip_file, filename=os.path.basename(zip_path),
                        caption="📦 All downloaded files (videos + subtitles)",
                        read_timeout=300, write_timeout=300, connect_timeout=60, pool_timeout=300, # Use long timeouts
                        reply_to_message_id=message_id
                    )
                logger.info(f"Successfully sent ZIP file: {zip_path}")
                await context.bot.edit_message_text(chat_id=chat_id, message_id=processing_message.message_id, text="✅ Done! All files sent.")
            except TelegramError as e:
                logger.error(f"Failed to send ZIP file {zip_path}: {e}")
                await context.bot.edit_message_text(chat_id=chat_id, message_id=processing_message.message_id, text=f"❌ Failed to send ZIP file. Error: {e}")
            except Exception as e:
                 logger.error(f"Unexpected error sending ZIP {zip_path}: {e}", exc_info=True)
                 await context.bot.edit_message_text(chat_id=chat_id, message_id=processing_message.message_id, text="❌ Unexpected error sending ZIP file.")

    except TelegramError as te:
        # Catch errors related to editing the status message itself
        logger.error(f"Telegram API error during processing for chat {chat_id}: {te}")
        # Attempt to send a final error message if possible, otherwise log it
        try:
            await context.bot.send_message(chat_id=chat_id, text=f"❌ A Telegram error occurred during processing: {te}. Please try again.", reply_to_message_id=message_id)
        except Exception as final_e:
             logger.error(f"Failed to send final error message to chat {chat_id}: {final_e}")
    except Exception as e:
        # Catch-all for any other unexpected error in the main try block
        logger.error(f"An overall error occurred in download_and_send for chat {chat_id}: {e}", exc_info=True)
        try:
            # Try to inform the user even if the processing message failed or was deleted
            await context.bot.send_message(chat_id=chat_id, text=f"❌ An critical internal error occurred: {e}. Please report this if it persists.", reply_to_message_id=message_id)
        except Exception as ie:
             logger.error(f"Failed to send final critical error message to chat {chat_id}: {ie}")

    finally:
        # Ensure cleanup runs regardless of success or failure
        # Run cleanup in executor as rmtree can be blocking
        logger.info(f"Scheduling cleanup for {download_dir} and {zip_path}")
        loop = asyncio.get_running_loop()
        await loop.run_in_executor(None, cleanup, download_dir, zip_path)

        # Try deleting the "Processing..." message if it still exists and wasn't the final message
        if processing_message:
            try:
                current_text = await context.bot.get_message(chat_id=chat_id, message_id=processing_message.message_id)
                # Don't delete if it's showing the final status/error
                if "Processing" in current_text.text or "Starting" in current_text.text or "Uploading" in current_text.text or "Creating" in current_text.text:
                     await context.bot.delete_message(chat_id=chat_id, message_id=processing_message.message_id)
            except TelegramError as e:
                # Ignore if message already deleted or other issues
                logger.warning(f"Could not delete processing message {processing_message.message_id}: {e}")
            except Exception as e:
                 logger.warning(f"Unexpected error during final message cleanup: {e}")


# ============================================================
# 7. Telegram Bot Handlers
# ============================================================

async def start(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    """Sends a welcome message when the /start command is issued."""
    user = update.effective_user
    await update.message.reply_html(
        rf"Hi {user.mention_html()}! 👋 Send me a YouTube video or playlist URL, and I'll download it for you (MP4, max 1080p, with English/Persian subs if available).",
    )
    logger.info(f"User {user.id} ({user.username or 'N/A'}) started the bot.")

async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    """Handles incoming text messages, checking for YouTube URLs."""
    if not update.message or not update.message.text:
        return # Ignore empty messages or updates without text

    message_text = update.message.text
    chat_id = update.effective_chat.id
    user = update.effective_user
    user_id = user.id if user else "Unknown"
    username = user.username if user else "N/A"


    logger.info(f"Received message from {user_id} ({username}) in chat {chat_id}: '{message_text[:50]}{'...' if len(message_text)>50 else ''}'")

    # Basic check if it looks like a YouTube URL
    if is_youtube_url(message_text):
        logger.info(f"Detected YouTube URL from {user_id}: {message_text}")
        # Run the download process asynchronously without blocking the handler
        asyncio.create_task(download_and_send(update, context, message_text))
    else:
        logger.info(f"Message from {user_id} is not a valid YouTube URL.")
        await update.message.reply_text("⚠️ Please send a valid YouTube video or playlist URL (e.g., https://www.youtube.com/watch?v=...).")

async def error_handler(update: object, context: ContextTypes.DEFAULT_TYPE) -> None:
    """Log Errors caused by Updates."""
    logger.error(f"Update {update} caused error {context.error}", exc_info=context.error)
    # Optionally notify the user or admin about the error
    if isinstance(context.error, TelegramError) and "Query is too old" in str(context.error):
        logger.warning("Ignoring 'Query is too old' error, likely from button clicks after restart.")
        return # Don't notify user for this specific common error

    chat_id = None
    if isinstance(update, Update) and update.effective_chat:
        chat_id = update.effective_chat.id

    if chat_id:
        try:
            # Avoid sending generic error if a specific one was already handled in download_and_send
            # This handler is more for framework-level errors
            await context.bot.send_message(
                chat_id=chat_id,
                text="Sorry, an unexpected error occurred within the bot framework. Please try again later or contact the administrator if it persists."
            )
        except Exception as e:
             logger.error(f"Failed to send error handler message to chat {chat_id}: {e}")


# ============================================================
# 8. Main Function to Run the Bot (Corrected for Colab/Jupyter)
# ============================================================

# Keep application global for potential shutdown command later
application = None

async def main() -> None:
    """Initialize and start the bot asynchronously."""
    global application # Allow modification of the global variable

    # Create the Application and pass it your bot's token.
    # Set reasonable default timeouts, specific operations override these if needed
    application = (
        Application.builder()
        .token(TELEGRAM_BOT_TOKEN)
        .read_timeout(30)
        .write_timeout(30)
        .connect_timeout(30)
        .pool_timeout(60) # Allow longer for connection pool waits
        .get_updates_read_timeout(40) # Timeout for polling specifically
        .get_updates_pool_timeout(70)
        .build()
    )

    # Ensure download directory exists
    os.makedirs(DOWNLOAD_BASE_DIR, exist_ok=True)

    # Add handlers
    application.add_handler(CommandHandler("start", start))
    application.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_message))
    application.add_error_handler(error_handler)

    # Initialize the application (connects to Telegram, fetches bot info)
    try:
        logger.info("Initializing application...")
        await application.initialize()
        logger.info("Application initialized.")
    except Exception as e:
         logger.error(f"CRITICAL: Failed to initialize application: {e}", exc_info=True)
         print(f"CRITICAL ERROR: Could not initialize the bot. Check token and network. Error: {e}")
         return # Stop if initialization fails

    # Start the background tasks (like Updater)
    logger.info("Starting application...")
    await application.start()
    logger.info("Application started.")

    # Start polling for updates in the background
    logger.info("Starting updater polling...")
    await application.updater.start_polling(
        allowed_updates=Update.ALL_TYPES,
        # drop_pending_updates=True # Optional: Ignore updates received while bot was offline
        timeout=30 # Poll timeout
        )
    logger.info("Updater started polling...")
    print("-" * 50)
    print("✅ Telegram Bot is now running in the background!")
    print(f"   Bot Username: @{(await application.bot.get_me()).username}")
    print("   Send /start or a YouTube URL to your bot.")
    print("   Interrupt the kernel (Runtime -> Interrupt execution or Stop button ⏹️) to stop the bot.")
    print("-" * 50)

    # Keep the main async function alive in the notebook context
    # The bot runs in background tasks managed by the notebook's event loop
    # We use an Event that will never be set to keep this cell running
    # until manually interrupted.
    stop_event = asyncio.Event()
    try:
        await stop_event.wait() # Keep running until interrupted
    except (asyncio.CancelledError, KeyboardInterrupt):
        logger.info("Interruption received, shutting down...")
    finally:
        await shutdown_bot() # Call graceful shutdown

async def shutdown_bot():
    """Gracefully stop the bot."""
    global application
    if application:
        logger.info("Initiating graceful shutdown...")
        if application.updater and application.updater.is_running:
            logger.info("Stopping updater polling...")
            await application.updater.stop()
            logger.info("Updater stopped.")
        else:
             logger.info("Updater not running or already stopped.")

        if application.running:
             logger.info("Stopping application background tasks...")
             await application.stop()
             logger.info("Application stopped.")
        else:
             logger.info("Application not running or already stopped.")

        logger.info("Shutting down application...")
        await application.shutdown()
        logger.info("Application shutdown complete.")
        print("Bot stopped gracefully.")
    else:
        logger.info("Application object not found, cannot shutdown.")
        print("Bot was not running or already stopped.")


# ============================================================
# 9. Execute the Main Function in Colab
# ============================================================
# Run the async main function using asyncio.run() is generally
# problematic in notebooks. Instead, we directly await main()
# as Colab/IPython supports top-level await.

if __name__ == "__main__":
    # This structure is more standard for scripts but less impactful here.
    # The direct `await main()` below is what makes it run in Colab.
    # We can use this block for any setup that *must* run before the async part.
    print("Preparing to start the bot...")

# Directly await the main function in the cell.
try:
    await main()
except KeyboardInterrupt:
    print("\nKeyboardInterrupt received by main execution. Shutting down...")
    # The finally block in main() should handle shutdown.
except Exception as e:
     logger.critical(f"Fatal error during bot execution: {e}", exc_info=True)
     print(f"A critical error occurred: {e}")
     # Attempt shutdown even on unexpected errors
     asyncio.create_task(shutdown_bot())